# Databricks Notebook — Auto-Generated

**Status:** Packaged under fail-safe (max revisions reached)

---

## Execution Plan

Step 1: Environment Setup [CODE]
  - Import necessary libraries: pyspark.sql.functions, pyspark.sql.types, and any required for webhooks
  - Set spark.sql.shuffle.partitions to auto
  - Retrieve webhook URL and any other secrets using dbutils.secrets.get() from 'prod-scope'

Step 2: Data Ingestion [CODE]
  - Read customer records from the S3 bucket s3://my-company/customers/raw/ using spark.read.json()
  - Print the schema and row count for validation

Step 3: Data Cleaning [CODE]
  - Clean the email column by stripping whitespace, converting to lowercase, and dropping null values

Step 4: Add Audit Column [CODE]
  - Add a new column 'ingestion_timestamp' to the DataFrame using current_timestamp()

Step 5: Load / Merge into Delta [CODE]
  - Use Delta Lake MERGE INTO to upsert records into the Delta table at dbfs:/delta/customers/
  - Match records on the customer_id as the primary business key

Step 6: Row Count Validation and Alert [CODE]
  - Check if the row count of the cleaned DataFrame is below 1000
  - If below 1000, send an alert via the webhook URL retrieved from secrets

Step 7: Documentation and Logging [MARKDOWN]
  - Document the ETL process, including any assumptions and decisions made
  - Ensure logging is implemented for each step to capture errors and process flow

---

*Generated by Databricks Notebook Generator · 2026-03-08 18:37 UTC*

In [ ]:
# Step 1: Environment Setup
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, trim, current_timestamp
from pyspark.sql.types import StructType, StructField, StringType
import requests
import logging

# Initialize Spark session
spark = SparkSession.builder.appName("CustomerETL").getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "auto")

# Set up logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("CustomerETL")

# Retrieve secrets
try:
    webhook_url = dbutils.secrets.get(scope="prod-scope", key="webhook-url")
except Exception as e:
    logger.error(f"Error retrieving secrets: {e}")
    raise

In [ ]:
# Step 2: Data Ingestion
try:
    schema = StructType([
        StructField("customer_id", StringType(), True),
        StructField("email", StringType(), True),
        # Add other fields as necessary
    ])
    customer_df = spark.read.schema(schema).json("dbfs:/my-company/customers/raw/")
    sample_data = customer_df.limit(5).collect()  # Collect a sample of the data
    logger.info("Schema: %s", customer_df.schema.simpleString())
except Exception as e:
    logger.error(f"Error reading data from DBFS: {e}")
    raise

In [ ]:
# Step 3: Data Cleaning
try:
    cleaned_df = customer_df.withColumn("email", lower(trim(col("email")))).dropna(subset=["email"])
except Exception as e:
    logger.error(f"Error during data cleaning: {e}")
    raise

In [ ]:
# Step 4: Add Audit Column
try:
    cleaned_df = cleaned_df.withColumn("ingestion_timestamp", current_timestamp())
except Exception as e:
    logger.error(f"Error adding audit column: {e}")
    raise

In [ ]:
# Step 5: Load / Merge into Delta
try:
    delta_table_path = "dbfs:/delta/customers/"
    cleaned_df.createOrReplaceTempView("updates")

    spark.sql(f"""
    MERGE INTO delta.`{delta_table_path}` AS target
    USING updates AS source
    ON target.customer_id = source.customer_id
    WHEN MATCHED THEN
      UPDATE SET *
    WHEN NOT MATCHED THEN
      INSERT *
    """)
    logger.info("Data successfully merged into Delta table.")
except Exception as e:
    logger.error(f"Error merging data into Delta table: {e}")
    raise

In [ ]:
# Step 6: Row Count Validation and Alert
try:
    row_count = cleaned_df.count()
    if row_count < 1000:
        response = requests.post(webhook_url, json={"text": f"Alert: Row count is below threshold: {row_count}"})
        if response.status_code != 200:
            logger.error(f"Error sending alert: {response.status_code} {response.text}")
except Exception as e:
    logger.error(f"Error during row count validation or alerting: {e}")
    raise

In [ ]:
# Step 7: Documentation and Logging
# Documentation and logging are implemented in code blocks.
# Ensure that logging captures all errors and process flow for auditing purposes.